In [52]:
from pathlib import Path

import pandas as pd
from sqlalchemy import create_engine

def find_backend_dir() -> Path:
    for folder in [Path.cwd(), *Path.cwd().parents]:
        if (folder / "iaCeleApp" / "database" / "local_iaCele.db").exists():
            return folder
    raise FileNotFoundError("No se encontró iaCeleApp/database/local_iaCele.db")

DB_PATH = find_backend_dir() / "iaCeleApp" / "database" / "local_iaCele.db"

In [53]:
engine = create_engine(f"sqlite:///{DB_PATH}")

In [54]:
inventory = pd.read_sql_table("inventory", con=engine)

In [56]:
products = pd.read_sql_table("products", con=engine, columns=["id", "code"])
products["id"] = products["id"].astype("Int64")

employees = pd.read_sql_table("employees", con=engine, columns=["id", "name"])
employees["id"] = employees["id"].astype("Int64")

individual = inventory.loc[inventory["count_type"].astype(str) == "individual"].copy()
individual["assigned_user_id"] = individual["assigned_user_id"].astype("Int64")
individual["id_products"] = individual["id_products"].astype("Int64")

total_qty = (
    individual.groupby(["assigned_user_id", "id_products"], as_index=False)["count_qty"]
    .sum()
    .rename(columns={"count_qty": "total_qty"})
    .merge(products, left_on="id_products", right_on="id", how="left")
    .drop(columns="id")
    .merge(
        employees.rename(columns={"name": "employee_name"}),
        left_on="assigned_user_id",
        right_on="id",
        how="left",
    )
    .drop(columns="id")
    [["assigned_user_id", "employee_name", "id_products", "code", "total_qty"]]
)

total_qty

,assigned_user_id,employee_name,id_products,code,total_qty
0,2,Ángela Aracely Costich Pérez,629,35531,1.0
1,2,Ángela Aracely Costich Pérez,632,11854,10.0
2,2,Ángela Aracely Costich Pérez,633,XT255R,1.0
3,2,Ángela Aracely Costich Pérez,637,8645,18.0
4,2,Ángela Aracely Costich Pérez,638,10631,7.0
...,...,...,...,...,...
11383,43,Yeni Yaqueline Cardenas Gutierrez,14578,17722,6.0
11384,43,Yeni Yaqueline Cardenas Gutierrez,14580,17721,7.0
11385,43,Yeni Yaqueline Cardenas Gutierrez,14689,8071,14.0
11386,43,Yeni Yaqueline Cardenas Gutierrez,14690,3131,8.0


In [55]:
import re
import unicodedata

catalog = pd.read_sql_table("products", con=engine, columns=["id", "id_odoo", "code", "name"])
catalog["id"] = catalog["id"].astype("Int64")

def nombre_ordenado(series):
    def key(value):
        text = unicodedata.normalize("NFKD", str(value))
        return "".join(char for char in text if not unicodedata.combining(char)).casefold()

    return series.map(key)

catalog = catalog.sort_values("name", key=nombre_ordenado, kind="mergesort")

def nombre_de_pestana(employee_name, user_id, usadas):
    limpio = re.sub(r"[\[\]:*?/\\]", " ", str(employee_name)).strip() or f"id {user_id}"
    base = limpio[:31]
    candidato = base
    n = 2
    while candidato.casefold() in usadas:
        sufijo = f" {n}"
        candidato = f"{base[: 31 - len(sufijo)]}{sufijo}"
        n += 1
    usadas.add(candidato.casefold())
    return candidato

personas = (
    total_qty[["assigned_user_id", "employee_name"]]
    .drop_duplicates()
    .sort_values("employee_name", key=nombre_ordenado, kind="mergesort")
)

archivo = Path.home() / "Documents" / "conteos_individuales.xlsx"
usadas = set()

with pd.ExcelWriter(archivo, engine="openpyxl") as writer:
    for persona in personas.itertuples(index=False):
        conteos = total_qty.loc[
            total_qty["assigned_user_id"] == persona.assigned_user_id,
            ["id_products", "total_qty"],
        ]
        hoja = catalog.merge(conteos, left_on="id", right_on="id_products", how="left")
        hoja = hoja.rename(
            columns={
                "code": "Código",
                "name": "Nombre",
                "total_qty": persona.employee_name,
            }
        )[["Código", "Nombre", "id_odoo", persona.employee_name]]
        hoja.to_excel(
            writer,
            sheet_name=nombre_de_pestana(persona.employee_name, persona.assigned_user_id, usadas),
            index=False,
        )

archivo

WindowsPath('C:/Users/lic_ferre/Documents/conteos_individuales.xlsx')

In [57]:
products = pd.read_sql_table("products", con=engine, columns=["id", "id_odoo", "code", "name"])
products["id"] = products["id"].astype("Int64")

teams = pd.read_sql_table("teams", con=engine, columns=["id"])
teams["id"] = teams["id"].astype("Int64")
teams = teams.sort_values("id", kind="mergesort").reset_index(drop=True)
teams["team_number"] = pd.RangeIndex(1, len(teams) + 1)

diferencias = inventory.loc[inventory["count_type"].astype(str) == "diferencias"].copy()
diferencias["assigned_team_id"] = diferencias["assigned_team_id"].astype("Int64")
diferencias["id_products"] = diferencias["id_products"].astype("Int64")

diferencias = (
    diferencias.merge(products, left_on="id_products", right_on="id", how="left", suffixes=("", "_product"))
    .drop(columns="id_product")
    .merge(teams, left_on="assigned_team_id", right_on="id", how="left", suffixes=("", "_team"))
    .drop(columns=["assigned_team_id", "id_team"])
    .rename(columns={"team_number": "assigned_team_id"})
    [[
        "id",
        "id_products",
        "code",
        "name",
        "id_odoo",
        "count_qty",
        "assigned_team_id",
        "counting_user_id",
        "date_create",
    ]]
)

diferencias

,id,id_products,code,count_qty,assigned_team_id,counting_user_id,date_create
0,13655,85,6406,13.00,8,31,2026-11-07 22:50:40.537966+00
1,13656,225,2494,6.00,15,20,2026-11-07 22:50:46.52305+00
2,13659,2088,11070,3.00,15,20,2026-11-07 22:51:27.592263+00
3,13661,1200,15029,62.50,5,35,2026-11-07 22:51:45.767622+00
4,13662,1618,3344,4.00,15,20,2026-11-07 22:52:04.398393+00
...,...,...,...,...,...,...,...
1052,14813,77,14434,0.00,12,43,2026-11-08 01:33:35.463875+00
1053,14814,577,12993,3.00,12,43,2026-11-08 01:33:45.992699+00
1054,14815,12097,20892,4.00,12,43,2026-11-08 01:35:09.423578+00
1055,14816,6161,1933,62.00,11,9,2026-11-08 01:35:28.036329+00


In [ ]:
archivo_diferencias = Path.home() / "Documents" / "diferencias.xlsx"

with pd.ExcelWriter(archivo_diferencias, engine="openpyxl") as writer:
    for team_number in range(1, 16):
        hoja = diferencias.loc[
            diferencias["assigned_team_id"] == team_number,
            ["code", "name", "id_odoo", "count_qty", "id", "id_products", "counting_user_id", "date_create"],
        ].rename(
            columns={
                "code": "Código",
                "name": "Nombre",
                "count_qty": "Diferencias",
            }
        )
        hoja.to_excel(writer, sheet_name=f"Eq{team_number:02d}", index=False)

archivo_diferencias